# chow_test

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/chow_test.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们使用城市数据（citydata），只保留 2011 年的截面样本，用于演示结构突变检验（Chow test）。

In [ ]:
%%stata
use ../datasets/citydata, clear
keep if Year==2011

生成被解释变量人口增长率 pop_growth，以及两个解释变量的对数：对数人口 log_pop 与对数人口密度 log_pop_dens。

In [ ]:
%%stata
gen pop_growth=v8/100
gen log_pop=log(v4)
gen log_pop_dens=log(v80)

生成省会城市虚拟变量 metropolis：城市代码 CityCode 对 10000 取余为 0 或 100 的视为省会城市，这是数据中省会城市的编码规律。

In [ ]:
%%stata
// 省会城市虚拟变量
gen metropolis=(mod(CityCode,10000)==0 | mod(CityCode,10000)==100)

Chow 检验的关键技巧：把全样本回归拆成两组各自的回归。我们分别为省会（d=1）与非省会（nd=1）城市生成交互项，使得每个解释变量在两组中各有一个系数。

In [ ]:
%%stata
// 乘积
gen d=metropolis
gen nd=1-metropolis
gen d_log_pop=d*log_pop
gen nd_log_pop=nd*log_pop
gen d_log_pop_dens=d*log_pop_dens
gen nd_log_pop_dens=nd*log_pop_dens

用不带常数项的回归把两组系数同时估计出来：d、nd 分别是两组的截距，d_log_pop 与 nd_log_pop 是两组各自的人口系数，d_log_pop_dens 与 nd_log_pop_dens 是两组各自的人口密度系数。这等价于分别对两个子样本做回归，但好处是可以在同一个模型中直接做跨组系数相等性的检验。

In [ ]:
%%stata
// 回归，无常数项
reg pop_growth d nd d_log_pop nd_log_pop d_log_pop_dens nd_log_pop_dens, noc 

Chow 检验：用 test 命令联合检验三个约束——截距相等、人口系数相等、人口密度系数相等。原假设是两组模型完全相同（无结构突变），F 统计量越大越倾向于拒绝原假设，即认为省会城市与非省会城市的人口增长机制存在结构差异。

In [ ]:
%%stata
// 检验
test (d=nd) (d_log_pop=nd_log_pop) (d_log_pop_dens=nd_log_pop_dens)